# 穹 SoVITS 跨句多参考短训
选 GPU 后运行全部单元。沿用 bs16 的第 25 轮 SoVITS 权重起步，训练 5 轮。Drive 每轮存权重和续训断点，最终校验成功后释放 GPU。

In [ ]:
from google.colab import drive, runtime
from pathlib import Path
import base64, gzip, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
src = Path('/content/drive/MyDrive/春日野穹_GPTSoVITS_训练')
job = Path('/content/sora_cross_ref')
job.mkdir(exist_ok=True)
out = src / 'sora_condition_results_cross_ref_short'
previous = src / 'sora_condition_results_v3_bs16'
full_checkpoint = 's2_v2proplus_emotion_resume.pt'
epoch25 = 'sora-sovits-v2proplus-epoch25.pth'
sources = [
    src / 'sora_condition_colab.zip',
    src / 'sora_v3_patch.zip',
    previous / full_checkpoint,
    previous / epoch25,
]
for source in sources:
    if not source.is_file():
        raise FileNotFoundError(source)

subprocess.run(['nvidia-smi'], check=True)
ready = job / '.prep-ready'
if not ready.exists():
    with zipfile.ZipFile(sources[0]) as archive:
        prep_files = [name for name in archive.namelist() if name.startswith(
            ('prep/4-cnhubert/', 'prep/5-wav32k/', 'prep/7-sv_cn/'))]
        archive.extractall(job, members=prep_files)
    with zipfile.ZipFile(sources[1]) as archive:
        (job / 'training-data-v3.pt').write_bytes(archive.read('training-data-v3.pt'))
    ready.touch()

for source in sources[2:]:
    local = job / source.name
    if not local.is_file() or local.stat().st_size != source.stat().st_size:
        partial = local.with_suffix('.partial')
        shutil.copyfile(source, partial)
        partial.replace(local)

repo = job / 'GPT-SoVITS'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                    'https://github.com/RVC-Boss/GPT-SoVITS.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach',
                '1b47ca07ef43a284e0b8bd7f29d3e1e3ff76082a'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchmetrics==1.5.0', 'tqdm', 'x-transformers'], check=True)
print('训练数据和 SoVITS 底座已准备。', flush=True)

In [ ]:
# 本笔记本内置训练代码与参考清单。
embedded = {'train_sovits.py': 'H4sIAAAAAAACCtU9a5PbNpLf51fgeHV1ZMyhJdnjxKrVVnkTO0mtL3Z5vNkPKhWLI0IaZiiSJql5eGr++3U33nxIM97de7h2bRFoNBqNRj+ABuJ53rus4KftvuCsveTs54+fT8/L33/9fM6uZx/r8mO+b5gsSNblvmmz9fM2213UnO3KlOfMP58FrCxYkudsFp59fwbgdcL+mjT7bVKU7LrM1pzl0EsTeZ53ku2qsm5ZUm+rpG74yaYud2xd5jlft1lZNEwCfKhTXvP0p2zdCpisVFV/uWt58+sHheqPpizU77IRwFXSXubZhWrxET4VSJ0UablTX81do37eJNf8RH0U+111x5KGFZUqast6fSnQ089ovU+TKNlVqpef6yQ9Xyc5r0OW7NtynTStDV8UCnKzL2i0SY5dvLOB9m2WN1GatIkC/gl+vy+TFNHi74a3JyefPnz4zBY0MD+ON1nO4ziIat6U+TX3gwiYy4v25M3HjwqqbCJeXGd1WURb3vre+YdPb37/8OuPb2MA8kLmvZs/N9P/PKkqLwhOgD8RMnM5n6wA07Jpax+RPmceAMcC2AtCNljxHKawmRW8tSCC1YkYL8jPPudqlOtyt4Ppt6uivGwariUizZp1ne2yIgFOxVgXsg1P2n3N5deWF7y2aq9y+uHg3PE8rupyzZsmK7YKN5Y2FYhgXW7rZBfTXADJUAS/Y6y2ihwoFzuuCU3xf+3zNvvI66wEObaID9n5XQHLrcm+8vpzraWX5l4KMK83Zb1LirXm0I+/vPntt7fvzwVAU15nbRPzXYmCpGD4Lf4Tu5Vx0yYtD1lWwI8879SGLAfhGmkCsqZq1mUB41b0XYN0I0+uuFmyv3/48c37+POHv7797TxEUmClqdbEmJOTk/NZ/OOH3979+jMI0/0Jgz9eWydZ4c3lJxXl5TbOipbX10kONWeT0NQ1nKdQNp29eGmV8qpcXzZQbhfmPKkLmOUYhIJD3SSaTCZTC+CCw3KCiuUk+iGE6tevVw5OrJvy08lru03Sri9jnDqo/MGq2FTTV3G9x6F8rvfcpqOOU75O7oiE169f//D9mTOg7Q7WqkI5m7z8wR7vGqUPB3bmFF5h2TSyIVt+24LA38TQnx6xzQ8Q2jReX1Ut1LxL8kbS+CD+8VDpuPOwS25jUIsxTMMe0b2Yff/qB6fPBvRfbrH4xQxYbDMly2Ea45wX2/ZSjs6qviwrU/fqpd3yJitGmxW0IteXSVHAciNpsKuxcrMjoZo41IqK5BYqfisLe4qSNI0v8qS46s9egZohueI1dvTCGd0aJKzgaYycdxoqlpLYuzwluXZofz2zOZKlKS/G6yVDrXqYEofaS56kWD5zSvPkTozglVUMgypQ8wnJe2HVVHFal1W5b3s8BI1+kZdrZJQ39QYqYgsrLa4XIfs+ZNPpagg4zfJEaBwFvpyGQAo7W4Vs6KeNZV+R/HGSPmo8nYSwKGHs9L9B2KzI2izR4tPjoAbsjmMGuKevNP7VAH/jLx0+7hsuDQv0WIBGd5eeWJaZO9+T2UtHPYAVALcr3oDJ4WqdebOzy6829zc15195/GWPwGBW+nJ8DfQBo7Gx9u08V1xRx6MqEtYMIdfF5f6C1y0APoD6XudJ05CHJwy89Ed8+W8wF3g8Dx2WBrTB1S9f2d/f/M6esV/2f3n76TM7P38PH9UlLL8dZ7/+1MDX+e+M7y54mqJFBrNnfE9yGQlnyjcsprmLY7/h+SZkqK/iqp2TfxOyquYgtVmtvi+rJkYQSRT+qZMbsDvC0UKr50sUIdsloIjApqEoLrx1tQeP6IZn20uwXWWR3y2QmYFGhAREqBmhP8Coukbv5+wUymHgngvcNPkA8MtTzeAO+BDq70+heF10QW0VDC3UuN0Kt42jlu02ToXbxuhqu4Ep7TBHq28b2pSeuOBZy3cNOpgrt1wAd2pQRPZZis5YnsHkkTSBEi4a9K3ikClfBVwenHMjAPgn24h27N8W2vlwAPAPYsiKPXcqcMIrIMWZ/eds490DNQ9Y4jnwOOcaXgmAAx9VbaeJ1UNz/YgGMJqibAVpUdZQKOBDRFZTMVEwUOwAP3L0qAeBNNET+ocQZjTCbXHg0n1NywhgwdT6oDSp5Sl7+TKAwfgDQvsdmwXdUU2iM/anhcH2J3b2Mpr0aTXiE0HMwovU90k2bKGAxV5s/cAIRrSB5Q8DCIJhfFLsFEYawHOgfQakdhYDoNA4QDnCUmZQ4RuyAvZn0OnAho33uSwhZLlh4E9lKcSIKQa1JPmbcl+k4Cd0mj442g8qpfKzdRqHCKjo9mk1q8EfIC8O7ZBUnTSJQlFaqG4yWK0YBkdlheggYCNAYJxXX3gBBqs3G3cKqvUOJrqoIgwLLvabDYdGmwg7JYPV4BcEnIX8CgBZ2t5VfAFtwBeavgoiiJOhwEckOC0vZign0tc0vEVukQnVKhy7jClS94GMINoXzZc9mkF/YqlqGDWuqG7c5jvDMNjDAQ0ZDmjasCsIYVfxgbhxdPYWZPB1d0F36sRgFOlIKMQiQWjRZAsB8BKnWFvBLL21ppBEH91RSwkurDWyBPCVwxsSBgUzICwjis5icJO7RvWwunusrY1SCM5gmkxHqLibPGouk4ovT6crVOBIpilxRZPEDEcGjei3W5ttNqyLALSU00VXKVGjP7MhNURceBdVyABAIRVMyFABYjP4idHAAlxfEKI1unFB1JY+UeaqIZ6rrv403hX8vYyiKGTzzhis6b2GeP5iaHKu/9lzI0UZiAqNVClJFHQYiQxRTMGbRIHGrT/ghdwn8Cm6lvIco3A3UFO3PNWjwE9f/HwPWv0zKXh/ebucriLU0/40IAfhFo0/oVuhysl2C5wI3sCaRD/TcugwykVxhVULvYBOIjXv+2i9bpcTiXbWRRugRYDSZ2waWIhwy2gY0yiBw5hw3QlEovlstLluInYArDYvZJtJvw01Qki0HBYeyQnpdfUYXUgoOcwjYHIMR6AU2QfATKewwFJLHCDUKRu/CMWwYLY0m1GF2jOi7I60HdLSaDLH8E5DezYOICFf6xhxEz2JoS13B9ASc8bw2pPu4kCPR9JFq6/pNxY7TeM9q+U62rvakTxEh8Z4ojz3jGwWSiIH240bttw369yyZUeUSVtDdQxUCea6hk3PxRK6m5OSRL0uFvJKqs+JqFopZe00l/K4zCS0auwYz14npInFBK+kdXEbuGgt8BMnynDRonq2sMKnA+3gtGA1kCVDhJMK1CrBRvjtQjs4bfCOdVEjoQ88gEAz5J9Ogx5TO+KkB4cTaY8Ov4W0gJWgyM9MMNloEiIhQUZuxNwbMyR7Ce25DO05C535CC22hzZTQ5t5ocMbJYtN2B0bGbhGWrhdcsVlk5hGAl6wxmB2k+doL9CH5Cn9lAshA2u15siGPGtav06KLcfmCoUKYcSxVvSJ/vERC0Rnl+CPQ4QncUhdB1/ljVo21DXGXxOzPkXUjP3AAsGuVPtQNrbWqKxaZjCRGRgwAUASQdPijwCE7IrfLfJkd5Em7I85U+L2x0pQKRmFs9/FYOheHaXXwAarETYBxtevXxtmyZ4DW55kmZzQmxpc6RgK1ldViUYeD8jUrhMdnEgGtTuM5bE2wtAqbqCH7Nb3okr4INKJln4NxF2+PHaBhoHCEKG/mKw59RJIEtxznqyAqIsXax5X7aVf8Dbe0vZXyOhYRElWyyv5s9yjRCuiza7dOdDQWCe/GrHyAUWIiCfFk1cg2NtszXC7mddIEQRYwM6LO+sUmfbvEHlZtcAK61BXKgYoX3oCubcahMApBlkJMWjGmabh0f4DBycWwURkY+9jgPvshnceDCL+4mF7wOVUAXqqrSJa23ojMqrq8o/oaU0gaNzFo02kjjyMVwINY+ps1fS2aeg8IoYWwEn4W4vOMLUXScMjPPkdrPUC01wSpRvIbysIdOZxqenAKYV50/FCdJnkG/gHAgtbANSpY5ImFQTMJAkHjjKFiFvt7dNIaky60j6PtICBa5tsS2CwRkw5CHtJpRAU0bp5oL/je1w4DyBjYiNKyPNFVgKkzETwe+u4xD1lgBEVdJoPCtebvPJQgWUl7ojQoZofLGfzlaUt1NI8rDFALQgtdIEU0DZ2X2EoVIE2RFmhVgl2pXIwojf1do9HkB/xq5ajqSI8E0tkle+dntL5ICgndO6Esqv5l30Gy5WCKVA5PK8WHlYBLxhtrQLN1A6jyzG8uMX9BLwIzjB8RfEXXTCzjx4ytQEfMrVfPtYxyvPp9mld06hAzTWzn80pBajd0U7wyOJ0eyrE2e1LIP9wzes8uWMJQ80AStRSu8bIYI4N6t3NPs8ZEm6yHQ6PLx3qc3BAPzkDYjeXvGCCRewSM2FKNwVjtFs8NTzO0w/7ttpjVkcNQy7rO1L0ZrzNKHp51C97IHsG0p3s83bxcpwTYMBP6ZzzSe1wMZ9ymKC7wXazyWSspTiPg47XVxdlgd0ma7Gp0sBoedwCKzzFi3cErBKtVBsGTopYRQy3XpjK9ZBaEk/OCtxMRuXHpPIbI2dfSILAK785TMzfcd73DYgEGfveUKBx3pRMYWSIkUglM8L8K84rXJMpX+viL2xTl195ERwnEJr9o/TBymiBX5pCRQmajZQlLZtE07Nb9v7TKDXQ4hR5/qhpQwY8p8E/7xgy6rUqwRpdiilESp6N0lHzDXq8gpzdvt0neX4X89s1LMnsmsfbutxXvoHtC112y9NTqDrNcNkrMfU8W9fUWcqxOa9pa4RolDuFxNCEYWZUDuNCbEDsHg+vRQtSSvsWbSH88g6Qsq7LpiFSdkmRbXgzqPvO6aSdCWCN1/TVwESjLLWXWcM0otHVuoNFMDxlogkACwaT3cO24DaKWHO92UIN5hLSPmnj0890v6saXyctyVgLYJfy5HC1VKlHK2FRm0h8DwBamUMa2JRRA3BIoEbkAP4C5jgBv/a77wBNoKoj6UwMwCxFBs/KwArlMQIs6TLQIqFzBFpksqwkt+SpIw2BuC7WV9naqZFZEyfXSZYnF+4pYxWBEJa1751zzPoEgfv5499YvQdndqdmKuXXmDeqtpzEp+8hXg97H+2G8RxsI+1cS1oFKAgPLKcYIz5fsyaiOFn7aA3HIKWf3UDDRIBQjBg9kFBPhpTGGiPBjdfLnlWIUaq4OuKThcGDyJZVrpJZWmjfNtD80tqnplUSw9oAIjF9Sc0E0aQrY7VMDMNFjq0CkKtYZRVi8SdVao4y7c5cGEV8ONZxyLoMFnvN6LQjf53cRxOnKX52khNwaxx3xkMHUGE3qXMIqFFYp3K6mcnjWmg4U2bgvvtOLwZRKI5pSAKtoeB+6FiCp68xRL0EIAhkRSLOwgDJkn5HQiELRQGItuJgVh/GNm3Nk52ZaBmLL2QFncI6W5aiGCblyt7JAxlSLWWU4saZopkV71x4H/+KkYzdDQUyBqcQd5pw+/SJGjzimImOTAMl4R7JDoXDCq+hcCDR1A4QCUSa5YpUGh3NDKXCdpvtMkoSDsGT0NsbC7kBIZJmzS6EomspaV1RwjPU2EMh2QE6KLqX2HGL60rscuD4VCmOWu0CoF7FnYBVN9EAK3r4lCY2RGPqQc0bsJNyEQLoDg3PXHW3uO+iWc7PVg/2yBf35jdVerb80KjTkRnSS+YI19JxrtGRqFkI6WOQW3JntXy08Jn9jINEaTsoM/C09w4zSuXKDcUcYF0Iv6mvTjKFsowqdmPg9qjoSQRk2lQkbjRG034gWlMjlPYVraSzX6aHMcTfI3QJr9GKJbuhKf7ZxmoPccHMtCsWdyU7Ke58e9POLBCNB7NS5IB3+6ZV2wEy3vBB8faCdGI/k0sBCgIjw8OrWnc2upr/lyTvX6bz0AXUoz5RJWrCThyfg7Ixt7HY2nCyLGWZ8nIdQDIXcuPKsUGm3XI+W43YIge5skNWS7WbJrnk5g7QjCgjZhoFTzNIwiSKZMt7h7j74a1UOZtzdmAfVsKEbraH2mk1GOyt2MNtLjK62+C0oLIHugBEe+rwVzBntBPpILE33feczmolL41sqm133fDhmwynumkyKOemvW0uHVJxt1uSa9nPoRxJ3BGH3uq2QY3lS/0yvgMuNurDzsa9lTK4sqV3gFRMU9WlXTHOQD/9jtx9S6p14/2slKbk9ZCV7nfyCDvd0wl6dx2ns7f3bg9KVi49nZ9OWVfq4F/lk4Ja8SX/VRfWlPjiZgqyT3KTfgbBEZb0OYKGLSvW5a6ChQrhntoxU526Nk65oQfOEELV1LRQgdyv4poARBaWPROh1P2AUiuSHX9g/r1iLK2x/8Qzhf8MHoKBaE6pVin/ql1oKTNNnaN1+66DWFjSHjcxXfPxBwzVFtt24VyKhD07CFJWEKdbjj19R2/SZPd399Btee8J4wNaiA5kDA1UznETkxJCvRzvK5i4zrm4xb6jfbKHlavnDkO7sHTPa9Huq5xbwT+VBi4kr0R0KCDgKxzI2xQcSI9zQHDTHuz/iTEYZ7rjsz5dpNIjMKgGKlh5lmqQHknHwwSYw2IHGji+0VbkmCF1mx3GXJc38S5prtwEpzjPrrhPzR0vxfXpce+3o8YktuX8+xfoxeD1qTPG/p2dRZNbmln4xNLZ2a3Yqebs/Sc83qFTHdzNv+iRtgRcc0Q2tTKi1dC2sLZ4HV+W5ZUvMypwmODULFR7a2NhTpVAyM7SeTgjYnu56RlYaxW7M3h83cKQH8IxZEuif/VULKunTIYQFiw/LAIGVggW/A0+dU05bU9ruQMlfryFzXB1yWBYWxLtT1aXL84evrXDAQY43YO73YOmQT+ZyJlNZG8ywUwMzeVRC/ZIK/Y0pnyrzfoGq2kT9VhrcWbf0PwXGLzjk/Nok/gYs/hU0/i0ER8ZtTtyd5/EHaneg3eB+I6ucJH0QFjp440MV3EGdmZmH/iYhXME0SK1PYxqfElbO571MYL6S920pm0Ovd+wpF/CA6BfOnnLbo6ylaU+FQVqp5OIeCZG9EwR9Uxwlo5UEdpd36uTb3BQe3JhrX5rMEcW+0N4CE9PWR6d0uO6xdR07/w/kZZxmfgfJOJbrMe3dv6PezCPC0P+X8cf/xLiRUY3rCl8J0Pk3klA9XAGLW3hpUbi6taCiRNfcY+BntaBluadHZ8XeOqbLhRedc0AMxdiff1JK2pzVpkVmHzknqPq2n4YtIlBVI2rDTrtJqnTuKq543bHu5Alc+Z3+sFbH+CAJ/Av3kHtVE5l5XSldKmOzmgYBGltH8nB0b0IvP4gj2O7d/dUsXN9r4e2e5nP1Ml7k6Y3CGgmL+m6hH3P0j6y7CNxwipgdeNP6UaNwWquwwgsi2FkQ1fZ1AibowP8prtt+jLDVF6V0Vcahsb8WElxFkxHbFzmR2KTn8TG3Gg6neL/A0ueXG4/ohUK2oBWURtff6O9S5H8Y9IFkKLnYhbmbIDNjC6uikcqmsV9f4J7m2D6Tivm14qbESL5vvtqQwGTQzlunZVjTuzFLW9xr7DRVxsQTecyUfcuSPcSyB38z735MXzlw7ldu7y1BMLcs5NEOfGqI8xZQ/7Ob87Gi3Nv0wZXEyu/rKmlaRXFjss+oM8GlKAZkphJHJEGEflJeKG+UQu0d1HfJAH1r6oqmfpRZYMpaaKcoSrJ6gYzYtAj+JpVsh9JRxA8jGyb4m0sx/nDm7fOHeuBd6vUhq+5hGpPbOBeglfPt/UNjDvGu/gyaemhMZIvuo6UA5NCdkurMWRf5b/+V/xdgRbCv/Jy24iPL/LjSyDujjSxuN5L1PpDd3+HJXdcVkciG3L3Kd+r98rZQLfU43CGjlXuvstkVXRuzuty9VJTryi5PUD2XSwIl2/GRcTyWCYDNT5UOlPxpHyhoD+/srfBN+L6rKIm+jb/dCS0HWDjOFyHreOALpvH4YbSo3pA1kMG40B6Ao+BJLfhwRmluRqf0ztrRqX73x3JyDz39JXynkfVryAopamvQ/1zS0/LxHJtpv6dki8x48pt6L8oMqxRxEZ5v2dxagXqF9M3dJf9xw/9IRK77wg0A0PTyDsXev2+D3byFLZJAMJGWtlHLdziCxYFX/R334Q/H9E/viYqiPAwAn0mfxAcfE/8Efuyu6BzI0pIzzrPKrHtRwfYsT8QCdEQhkkCh8Tg/zbL8I8J2gZ91Vr+ux0ROZf2p0kZ8VvotXdRPlUSRapTK73ACbTpIcJhRFeIR7626X/VBu5Lx9oJW9jFetVH+ljmaQI2mJ1oPwbquxwMxptu8cY6LjH39VB/ZE1128b48O1Cf+LuuCTomWHyM+khqLKBIQ+v1bGeRpB2t8AetRb761B2NrgUh5fh1r7mfWAJbg8uwf7y2/Y7rlI8vQ8e4+q52OVdXhHDaX1j9HZoV0kWjNQC+y19f3LAEX6EGx5dcHwBkK4N2Jf2Bb9olfjdQ9duqXyxkLx1MqLOoypLGSlDSBiq7QKIAu1s3RRWKXAslx6hichszEEv/xRfEdtgIgT3lzaOVRBhTkgwmGE4nPtoJf5VETn11tIXu75UHCUXDb48tt/5+LLWRGww98+XcROxkx6iL07VfM2za7zcUtJpaIY+hteLhLu5/M/eSkQifGl500IM0+BVJCi8F1bzYS4t7OI+T+fRy82DXMHwvbW+gUtQAn9T2UikI2T2RF9eicp9G+2u0qz2xQvPjbxTx28hhorLK6u9SHOkG6AqDw9a4zuCzSy+nlV1WUGPOitGgOv35TCnEJp5cmkZeBM+Xk89R77l1YteC9UDwAvC9gU4dm0LjmrjpNB5eJPIm1tLA2Rry2uaj7EneDvXaJwnMfHKmU5uULCdnAf7sc5CNqFLcvOB0+UhYLywNu+fj1mgSuAVmPqWj28OMFFvS1K+mtANePpPZdu8vMCrLLBIMZFAbo5ecnyzs45VZqK9s2mJQkSS4tyWb1p3X82CfnpiJD7S1YoXxlGEvADzxUiYcAmbnVFMHesALo6K29GMMX1Jhbps6BEtMBrsbwCX0G1Jup5q3TptS8FjWQ2yaV9r749mIUbTc/bMwOg+nWxki7pghV3St/f9EXV3TnCfXC8eMTyRlIaXg/G6ampdorWHMp7k3LT60kLwiLR6BZ52wZXTMdRA1nWbDDt6DgtlPXDvoCuooo/xznv0umsLAem3t8LrRg6ku+IQEn96q07w0F2Cagi6pvGCUbkZeIVTq8EcXPqYrllK90dh1hWwKu8fOrGnMmOfcD3jnu6oPSMBosEv7i2uPJAjsLi3hv9wMJPxpKOLumewOfBGVUZvbyuYyQKTLN9/Uv5eyLbJbpdYx0jq8XRwLHCwgspT6SeZrtKndpU+vqsxMXVVNeoKVwQsFWtY0pdQt9HSc1aJg3VglXQbg5Qf2Fg2mI6TkerbnuhhiWWiX9qxpMTeeBF3X+khu2/3genvY37wY0LEYV+5H2kJOA5xim8nM6gXiIYecFJnT3qv1ToZNb5I5yIkpjC4Y6N7FXimaf4jH+Z6ZbHfxTdlTTcWZyGrMtwH3IHRkj6f8vI3xcJ1+PW7R+S71wvnSSO57xvL/7hG72Flc3aiL3PVLrvGgwXr+Ka7zyz7U2lW2DYY1bHPFh0FDLNt1/+HfXJt/bci0DpPxk4hBoPrjbfsasWV0oRCBz6/74r3kFpk3gh6OxZ40YkFXgzEAi8wFhjGpdVuvz4Y1DQUSD9GlYxriQ6K5Hord40w/jLTig/24ouTU/EAl1URdG7jPPLNqtCe79COXjbgI9bJqUBzqh1F8SyInDPrQZZ/Ys9Ox7jcwAq7HfVeBnPdVcuvdpNie/MpvM05uZr92XbdyrnjUw5AGydhPuRTjIQfA4gsP2bO7sFOzTviZt7kguA7dapTp7ori7rXgewd6ZtCj1dzdm2O5+kNKXHPJGTXh58GG0ObPhZt+mi0ytOda3/Y4ctYg1Q36HPqoO8xgFG4sXMpzwPziM7r3JFzB+hhJAd0xIqPHfKmdXKDx7s0ECN463JftM3o4a7AZr2+cQ/0Jlv6r1F0o0O6uuUMdnBwzJN6i5QSLhh8bd2XhSF7GTwEndyEkwzf2sZLPXFMgWkc4/NZcSyvI4q3tE7+G+zXGK1BbQAA', 'cross_reference.py': 'H4sIAAAAAAACCp1YS4/bNhC++1ewuoRqvMpmm6SBGwMFgh56aFIki14EQaAlymZXllSS2kcW+987w4dEyg6S1lisZHI47/lm6CRJPvOWV5ooduQXauDshksiecMl7yquiOhqPnD41+n2gfQN0QdONJN7rsmoNZcMyLIkSVarRvZHUvUt8hN9B2ePQy81ed+PHRCuSc0bNra6FpW2xM3YVbrv24m0lWNZserA7f6BqUMrdn5XHdjV6zcr9+1v1Xf+HbSo+6P/dsdu+cp/0b2sDpadec26Lhu1aFUmu85zHlhdKv7PiDY7QwYum14e0TxP9dsfH69///jh82q1qlqmFHkve6U+eWdtVgQ+YCQpS9EJXZZU8bYBu5lmius1AW6i4UqXA9OHNVGc16k9hR/J7sg2pskkB812D5ormk6EngSo0QlZ27NaUTg+k6DgrBHdnstBig5JrfcMWXbg97XYAwu6OOJUBXL3Fm+jxsgKHvGG6Jo+UD5PhOZHlRQx1SA5sFQRoVtbkvJjb5NoSx6f4q3dQ+l2Uc05p2grlF7YU0FiCDCFI6N8IaPCxMQNl6FLZ0CMdQm1sFDCalyKWhlXaBralt2ydsRozbxstTj6x1HUBBKLwHNNSviDEvPOzozXZkmiCQ//sJ19nd3wBxAyJ4/JC6EUhBy1gnTlNV2Sk4uAXxqd5fdasvlkIPbiRGp8UjKhOPkLzf5Nyl7SJrmGtQ41QbsIhGBO2Vo0UC4br+v20b3km9fF09qqsX00D7OUpKtJWui2qu+0NMix8F7sEtnfkcBtORwvIgKojiOTD0DkOeablnfU13paZGo80locty8XHpty0NPmUGjUMczAgUd2D86K5UXJbdQBBu57RAix73qNBuSJBn8kBQHzp6UJopMithg/aIroRh67gjdB3QCTDCJMZ0Z+M1l7fdKlQhGLLUk6PkKc2sSEeFLNrZb/XcU7oQ8GvbMeeg5VWlIfW1gtayHJC9IkWENPuJKka5LIXZISpggba9GfCqpHyZzNhgKt7hoJ7Q4L4sW8aNaAltN0UVVdGTC5yl6jK7CMIfUCLOAtlMFPS5dFp99tSfj+8vJU3QXC5YHLi4wN2IopyE7PH5zh7iytq7VyxuFZ/8xWJsaLLnilq8igiMfmW1jwp5PVMNFC85iSgqhKco4osSGPDnUWvFOs/riBMCENikZSwcaN3S8HnCaoAQmfxDNs+NSFuEVlaIEjCHuI9VjTkbQmMW3/YrZkx7obMAJhY+m5JxJ0IFRkpjK2AIGFWvULSRZiXB1t5yNBXjxzu8+K9On06KjYruUuumr7iBDmMnbZFp13pq2TPvaUrGPu7agO22s58nndgbSZfWwI7ORjbZtjsVk2RUS2CKDteuExb6LnrdgLtAna+DL4JzbM5i3LEamgj7oBFmEr7g0T1k5ERsWJTbFa4LPX65t18KGH3qcA82DYnlOH3UJZmFiZ5LAio6aHPh3MgbKH2VuWUDdi0BQ7eCw03D5x0xlXAYcTErBq6RDL0LrkTJz8dsSqWHQePcou1g/UQPkn8Q3GKr8UYAAgtAWtE3fk5/IgqBePoQF2hyKLWYgrrP8pZ+qJ3yHJlppPf1942JVPGrBzITpgTXJLaiQ4HoEQ24jyYu0tWZ81ZPbuVLo7voduxYe+Orj6Ne9BmhlFpRkx7ZUr+2QedL4dPHeHzo3aWdVyhmP2JFKZ22eEFqE4Y641MTDH2DG5DWHUp+MML2KmAH0zqy2NfA8ukeY6QxMUBHMPPlIzdePxd+Qye2O9Sd01JfHaxFRvPZUP/qRsGg2vqh9lhRYBRGLI6KTFeqHE+iwzXDXGJ84J6QICWAvXmA4GjFt74Vkkq5U7oWB8yTGJ50JQLHEzZHxmmDv0ouKhv+0KDY99ZWSZ5PsY4ihuTn9lxDGZlFtXFuQ5jFEnhK5aFmzOg7JXkX83Kv86/UpBYcJX4gvfvnmVzg0Qr+Muo3H6CtJ54BVe+LbRXTsrzSUfB1wkoNHeV8bemeUtcLO/a0xip8PqzNlswCQ+sgHUrMwkuk2qYYS1Oy72B63KvmsfTHs/RR9QL9/8fPmqyMz4vh/7ESaE7BqS9jaruQafwFcokAMbOL26fPX2Ms0aUEyHNb9jegIYN/zgb0O3EKvw1xAYfG0vyB24GKRwCWoC47o4JLZjU0SuVjh37Hhdw0iJfL6Igf5oPWRjhNEJy8OJDO63yAYbQfDzEHWsjRVlA+CjrbcyKNFODb3i9OWaXKWxj3RPnYkTcxjr9vqgpgBq3sEcTPP7DHOKXlrd7k2XQZmF99J2yenI1M3EhoEae6umZXSVLg/mH/oOYAj/FwBgTpF8Y5fcxjL61KYvykJrDP9aPww8Bf5WtIIMuKGz00Oz1969q38B1rR4vnUUAAA=', 'cross-reference-manifest.json': '...', 'performance.py': 'H4sIAAAAAAACCp06bXfTRtbf/SuE9osEimInpWV9UPekNBT2AZJDAl+8PjqyPY5FZEmrkRLSLOeEUhpeSml5aGFZWMoWWLaFlDbtbgqB/pdurCSf+heee2f0MrJ56XkCx9bcue9z5869I8uy/EfLt1xCieR3Agu+mp7bskPbc213Rmp7gRR2iERO2jREwHuT00NT3vGD01PS3IhESddyQ7sptUjTa5FAl2W5ZHd9Lwillt1uO3YjHZ6gnps+e7TUDryu5FthB1CkBDwJwxQlIOkTXaDpYxQ4gK4H5M8RoWGpdHRiYloyGJ1imm3bIaapwjT1nDmiqLpvBcQNS2OTkymWR3XiztmB5+ozJFTkqYmjY8cnDu4bNwFJ1iR5f3U4N3HY8n1ZVUvvjk2PvZ4DYgELVCqRLA1Lcnz9Xnzt3vbyJ5v/+snc+vbR9u3V+PPH8aUVZDx5dBz4on6ACXLNRK4fkDCwbJe0zC741aFyaaqCGgA+YM7QuaG5kbbtWs5QjjpMKw0ShCO7O+8P7Z7tDDmeO0OCIeJ7zY5RGRmiIfGN0Td//+abe/TmrB8Cz5HX8xx5b2R0dHRW98OOXAq7PlAwtwMJjBhI78627EBhMWJ6s8Z0EBG1JDgq8ltWSJTp8cOTBg0DBWhUTZoujA7sNw9MHB5nEBQwLDetZofIydSxd8yJ/fsPHTwybsgV8PKxqXHz0MF3jh035LIsCqvJk2PTB+Q66ImswLfDchBBkHaBl7QLYk/HsKPE54M+shKEG0OoVcvIo5YxyZcHdXoBeJhA5I24JBTm1Xoau6EXNDs87NljGvSuWyqNH56YPjhxZArkKbJLInC+I2uyhesH350oCOGrTSwcnfAW4JN28DMkbouh0CjwA5sSeEzCH55AghXaEIfgoH0Hxo4cGT+EIhziKqlEdecIuOHN0qGxd/jkYkmCP7n30+r2rb/KVUEbiOLe2loBtH1tZfPR1QKo9+n93sqNIuij/8T/uSOCuIzNq7fjx0id2hlfuBZfvCICPrwbL4mAjbVL8dM7AmDrh6/ipa8LgAegVQ5IzPnn1d535wCauDL+4PuN9bv5ePPa37Yu3s/HW8tfA+d8vH36WvzR83zcu/vd1qfPBH5nLsZnbmbjROjKarz0OQCThdv84Z+99S/zce/TTwrz8dmPN376ThhfPNv7+Uw+BuN7K+eF+XNXeneXs3EidH2JE/Ew2Xr2tPfzN9kwfvhV74ub2XDz4Wfxh5fz2Zu3e8/P5sOnd4TZzKbN538HGI+/zQs/bX33TTbc+vn25hefpUNOsfHk+/j6jwDLYjVe+1d8+6oI6Z1d5UpnkI3nt+LTFws4K+fjCzcKfK5c6T1YFiBcYHzmwtbKv1GJfE9s31wGcBEGmsZnPivCwN7e3S+KsN7fHvfjJb64dYOHer7jABR/fLoA2nx4ZgDro9XNp7dEUOKqtUtbS58CPN+4gPzBww3mGxEYL93rnVvux7x9MV56UASWTpVKLdKWXC/oWo79PjEdq0EchX2qVSYXkn0UuJJLToaKMmc5EWHHPhxhocaHtivxBKHbIelSRZXsNpvHGc5LAgoZjjRNOuK5kP2ZVL8Dz1QJgXMiSzjT2bAtpfleBR0ZvzQBcwL8SyG67VI44JSyllOpDIul1TmvaTlm6M0Sl6aCjk/sGztkTo+9Zx4d19LRxP+MHzEPvjuVk6KGetMhUAgFKSkbmjhTxCvMwwGNMJBqUqxK3CYpMXQvCvH0qHPaxJ1onqiRTn3HDkX/JE5BVyBBDsQ/qMygBouIiNnHNjWtSAjK6FDKwDZR+vBqSF9XM2ziJDx18LDtK2qREV3oNjyHapIJ/8G+3EcKCxdJPmFhFTU3IqsDGgAWavAyvykZc0auirEJ5DyiIB5tVksgtSb5juWmkcW9xlYy0Z3F5J9cGf3DVg7GePgxf789Wi4jAIkqew2EM257jT25zVAEQUV8HDfBeBB4gSKPn/RJMyQtCSJbykrnKAxJYIERkuW2pMovS/+7h6/WTORFVKKARhOPNKOAglhDKmehgdOoI8ovhIGCMzWZhuBbub7DSGgTrfloL8cBz8p1bgYzr+B9tgJB5p8ap6wKlPXUZ30LPuiAKbRF6kYUFPDmCG8PvMCewcqRu5mctJqhsyCEAFo5SxZYupAhiXatgBUxBJsNfO4Ti3kBlQOaumRTpjnmFebefCbJGGkxUx0wekD9tnzMnXW9eVciXQ+bHGkxY7cjOCWonMRTea/B3WS7EL3UDhfQzZXUndm0b0WUmJRNvs6FB10WxlLGUmL5ARgI8vkaGcIapWHOZ3bka/2qeD1sU4p9G1sYC9YGHOi1WajJSZaG8gE2WZKnC3uquNvYBA/ayHEMMbkzaMPxmrPUqAkTNZpFb5WmoabyoE8jntt1wsN2A4nZLOOFGJwpz6CdDFAXtzxqk64HZ/MqhxzxpIhaDUfYvUzhLkl3KEQHCcAOm0m1tUwX4kZd2ObgD64VN8Ts06prhdC3BEbS/+pTSX47zOGKZXAltYaBmmtWFHonInfW2G85lHAVLFi0GbcLR7ghHCGej022ZmkNralh+KSysBM1+SxViicJhxqGDFpA6V0MzVxMmpy57TWr2ug/FFpvN38jcdd2FYxNPlSHKpq1C0JdObGza51UKlpjyFKHldZQU1XVOjPsBNoSYMXOwaXktGsZrFXS3yeBR5W0hdGQObouPfzZKqVJNF8kIZA5PwcW9QQXqDHlivi5Qay+YRiGYdf7UwIyes3BnOQ7jcU+y3pC2iurmiKkPq3Qj6mvyISD+Q151JiQXSkHKJJa5KSSEaka6ls3KgXlYMUCi1VbgMcvMqwWv/aBGirJjRSO8lp9UCFOnCTf5IxKratryZmV2dfPINO7YHe//kzGoO6DhDtHEqSBNP1Kol0VRlYbqtTr/SlcrDzYFkUGpVLTgQiRJknQxmIadvRYy/Lh3FdcVz/stSKHJJZiUjVN27VD01QocdqQVAPvBNQN4FTBGzTygVrVM9x8yyGVnhNBuZAP+pByfQALVDkEycUK8t0iUOpQR5lputekhm1RMevgHzBAZfiW49qLIvR5Ys90IOFnhsLMvBW0EjvR+V4gnkjcjX32KAmetGvAiGSqpuu6JlVTKyAb8cMKWoDQchwluXDMPM6vH/Fm0AbbzJnAapmKYFuKgTdkJiqCN1+DSzmAllLPAf7ApGBR4hgdMO0mTwStcMEnv5UMcV+saMH9oadwCQZ8aVyEwT4L9fIrmXBHOh54yHJhF1tsr6fnhm/gPeawHH/+Q+/yva07D+L1q/GXy/HqA/lFQB3vdR1ZFL4YwDaEg74asFQT8IPKVxBTR7FU82GlLN4CKHA2ei0oUQw5CttDe2SVd0WOjUWEqkIDO3ng6NjUuDlxfPzo0YPvjk8ZOScluSvkd9dDWJAGNpyCTC1ZfbmYIcj12GQwXyRugKYEzwwl8OYTZyClAcOazLoVaG54ckgSJJ9KBslUkgiN/o47wRIyJV+xLFG+nELMpVnbnJEZCUMhr2RzWDMz8O+kfVA6s2IOEoJHk0wvMUlUsgLoYSjrapwFab5DoKLxwo5E0ZdSx5ojUou9AGiG0EGdtKHRliAMW1jX6NyMeRtLEWosnsoOZRYNeLIzB8L5zsWazVQVyLW51taAA9iNhI3bEjnV5AYBrgScoLE7hjxpNV5LarXh+QWU4EmLNRUN9mntMBp9HW9iV43z8dptSqCarRsKlGHZajALMebR3P5wFUx0mx0v0LgdGtPJ6MeuZbzyA4xLNVhvy49Hzim3A04mgi9WGOLbZWYMv9TwIii9EnRDOEkzwzgNGCSqlcRm6Pmwon5udxWeIdhmZV64+fny4v7gJ6i4vDwYGrANoTOAtA57DE6KUEmlq39Z5Kx5gcseNRTAahMUL1454cQOQ/7v0mn5lCpyrvm5OhyE6OW9/t6sScpcUqv6WbebA/1qBqzXqm9x3/O21iinjTu+00m3G3cQNsNpif476d0oYMlUYtdmfFvNgzZMxFDDwh0GxRqUW3D64kLilQy+J4KOLILtCOUnYMzDdsNczfrDAFhRPdtS2L5lNu6qZdbVB66PmFXcgCpQ1V98oTNQuUbQL3QxTaYrxIpDYKAKR/oM5r4Us1auo8x0CN0CvkTkzspocOUgANiCJgw1Weh2WeQY+uhutpuITqGEaXaUQK79snT/v0sf/LJ079f107+u397xh7rMAgS6LyZJr4ykoUE5pJy7Ii0GDX3PAGPgtwN49fspZfvG7nyr4CqnF2gtuwnhiyW3wak0gBpoT5r5uX+0PAtnT1wHnN0hAJk8zEtarm/2pCVVqcG+hcyVxCYIFlc+XYW+VU5CN1uySqHGZfYlN7Zw/pmw8h3TcxyrayV3Ahgm+U1DMwoh6MvazlpXR5fwPrjLcgHR2za+P4bMCz6GlUuXDTbtr+sf13f9BVcUlrO+OKKd4u7ne5uzyjdsXcsfudgOdMkgmK0YdqhMLKRhFPw+xDbqpeFHrVKtq9kdAbLhtOrbwt0es4JhV/fUxa2UY/x/BFKfNI1FGUsyuSp7DSz0ZA1OfQ9KsNCGpFhdzCRkxUB1UcZGVK7WWCDsTDuV+inxluwVWDnLvAOqZmoAUYO9M4HVt7vIoQzP1kn+XBHJ0y7otxOL1EkB3gIVX3jPpwnqaXnHxY8JCvmma73OecBtBpt0UUUrCKwFruFBPC7kqrDoWrH9RNVfgCTbHIbrByYVLMkkJoqCPl0/NBR549mX8fVv42v38MX+2Uvbt1d7365DbQxxH398vvfk8ubTv8bfXt5Y+3rz/HK88uPG2hI88/p5a+VO79wV2BMb69d7Z89tLy9vPlndWLuwsXYx/mJ5Y+1RfP2TzTsrmzc+xGJ79XPgiYRnzm4v3eitrfWe/nvz6aPeuceAD2w5f+C2fXoZgJx269nD+ML1+Pyl+NI/gFySC56QpzskiXFe9zleE05tLA+pjRdkYCb1WguAg1czcGThVW9+1T1vO47UIBJdcGGCQvEFBRRlF+PJsYJ336E1S/R+yRvPL/66fgOs5u9vN559xN/Qgv74VvTWc7Cod/mr+DF6sgf6r/yYFdbsxa5Q4+IrVyHTl/W3gAu3HCilNH+W9dGEu8CLvzkUa3C8SxSZje4GDTbvn44f/QN1en4xvrm0/c313vqT3pP7L/DpMcjl2Dtj4kF3NaD+YlmedrzIaTF/+fiKSvCRs6BLhywacl3xvhs56NIfpyaOsJXR/+QWxexiHVAr6vpUWZST7CxXkwdN5qsqV9lhld9wcbBWga4KTi0aBcS0aNO2k0sAfrj41gK2VgajZb8+MV7+m5fJQ9ifH4Wt9+d54o7ou6ujDVnVkg3CvzTWeIYG397awGUQvswjVpcroc0S4ptQCs0Ro6x5PutNMSWQro9GgM4s/4BN5owfZc8+bFVQWK6+VS6fSnth9jMho/irIf0o/1bkThj61eHhyshbehn+VaqVyhujbwxbvj08Q1zmMTnXFgpayxDcnrjpRX7UWa9JFFXrQPuJd8mL8j5cGjccmk4Slg89bpMVi8OsVU10xqO372dOOgwh/UFbysca/pTFi0Lj97tV3G8BoZEjvCnEI1zskLNHhWOqNXzv7INbsZUSk1vhhZQmnHPIkp9z1aFKPTvxNCwpmyH/vU+1UC6lP/nhhZLFaqSkXUJmeO0mXLglFfbLXjckZQoC/g8S00XVsCYAAA==', 'sovits_emotion.py': 'H4sIAAAAAAACCtVXS2/jNhC+61cQOklbWU2Mnoy6QBqk7QLpIkCC9hAEAiONbTYSqZK0k+y2/e0dPvS05RqLzQL1wZZJzoPffPNQGIbvuZZ0poBr4DkQqIRmgpNc8IKZJ8bXhBa01iAVWQlJfr65m92K397f3ZLd/EaKm3KrSORXOOhsHadhGAasqoXURAuZb4KVFJV7JH6d88Ct1iBRbUWNdb93+cvFhw9X17dBEOQlVYo47VfOtzt40VfVIxQF+hZxnv4qim0J8SIg+EHTF0WhyEeQYsY43oGW7CMUxs6s3ggOVXdLaNQoIrgWxKrmuShAphqfs/aAvZI1UMCKZJnRnGWRgnKVkEeqYEFaTxKSbyjnUKoFYVyTZXsh76P5qC06FMVpqyrutlBpanSipPkZbtRS/IEbaO2acaAyaow5P9LW5axgFa4xqpY/0VJBZwBljdHUYKTcJaza9BnYeqNHnmTIBS1FqdDqB8SvgwHj9kxl4VEwgC1ckNM74ErImMx+GCx01xdbg0t708gId3bZamyaKcKFtvY7JebTc24okmoRFbBjOSzRWOoeE1Lo19qvmKc4RfpzVQv04Twh83ig3Hlpvr/pwI8aC91ZCXoruTk4ouztzpP2wqXQ6XTdajxO+ymptlVF5atj6roUj7QkSr+WQNTOEPV/QFHyjsw9TRGrbAUUcQP1RVjawHOcpA6qz6apEz9A1Mb6JE879wYCJ7L0BFZ6hROkNEgwrjQty0yJHdMq87yKbM0+jRsGqJIpfY+Hb6ikFSBJH1om/y5prVwPSLGbZPWoiBLKC7/tgCTPTG9agjdtxhLZo2vAZMp6jrkQTetOjjSJHqcRYMPMSTWeXD4QnZiJwmmC5mTH4Wkslke72pF7NtGJDXcGxPBEwp+GQUNv4gbVNWiqtXRmEhIyle3myKIwIS4JXaQOYe8Cl0zUtwmkfYE6BdvR0QNoeuosJ0vswM8BWhMAxa5a1IbRppEYih8JgatCdcN/FcWnA9vh46ylqBl4MXB6Sr9PaCfoc1qhnM/npi0dSmi/Nax75C9bp1znzkrga705fMQX1a5n3YJWg4mqbcP3PybkMiF3D5ZCXRdr6p/Zn18+YBdz4TyY9f22PyyluPkly8HxHN0ffXpunJZDn58/Y8LvdbhhnQ+CN4DnRGiaxyD4StigiT5np7uuzir60vMwVThcRaMpr6LqCc9EjviYXXyNA6mRTJqS2sq7/3G65erPLQAqO4vJ9wNvenvncTz4t2c2W6Fh82CqU2dkv+dXQHmWGy9b3N95DXGKxIhw2F/OY/KtX+ytpTiQVnVUMb48T8/2fYAiwxlpDFSzw8oy+sf8ScjsHL4bitdAn6xXQz0o/NIY39FyC6rnw1nfByzMo4i1F+08wRWv7bDxns8js8F/JZILeU51dO8MJ17tA4YeNWHIXJ2FF/NmOhqdMmSvhr16a8ekguW6LZdXL/iWkWPJHI06xLU592Kt6M7MBZgTrMBXYb4CaV/J0xpnpHwD+VMtcB7rCqW1jpf4FDZNLly0s9rfb1EPrMX70MqaJhU+HJ+JbCOzQpnBI4q/VoHwjuL5kZv9DrvnmG+wdt3HvRS0OC3qiZNb2Mj7vtk1zGvUo0yI8dVuigTMvNhRXC8l0OJ15ud1aGbm0VjszI3bgD/QixHq9VQ5iOnbtAcLsAOvQ3mfPi0h2lANvX1rnuyTYsLnlklx8C8l2ioVNRMAAA==', 'vocal_tokens.py': 'H4sIAAAAAAACCu1XW28bRRR+968YbV92qbPNBdpixZEKsUpUmkY4qhCWtRrvHttL1jOrnbEd01Zq1RckkNoCKrz1oQLx0IpKvFSiiP8CJBVP/AXOXPYWp6XimUixd2fO9TvfmTN2HGeXs5UZZAOakBkPaRJ/RmXMGZH8AJheGkwTmi0IHKaUCbXl7mwLcmFj3fcvbFz0yJBn5PLe/kqXX9/Z75LZ+rnZ+l7G95Kp8B3HacSTlGeSZJA/SZ6F48Yw4xPzSOw6Y43Ge5e6neD6tfcvvRd0dz7pkLZy1GiolQ+D/WtXOrtdXHMbBP+czREV6ZbTVC9nlGCLvHz09M/f7h49eHH0+JkVGmRA5ViLKaGNFjl+9g1u//3iHkor0fvP/7h95+irn4++/dXqjHkSVQy/jTpf3nn5y9Ojew9KwzyjSUXonRY5uvf4r7s/Hn/xqBRC0GRF6DwKPfzu+M5Px58/PL79vY31/nMrLfjACmvpCyj99dPSWEKnI5OJ3r6I+T7RAXmNxpWd3W1EyICEGN0wOlVAnFaJmdldRkLLFJAZKQPNaTtVVPS+Bc7qVeDQuxYxu3saDlrMYmbFNAB63aBj83pS+MxRadyqESXY2VZcuYFMbpGTvDpLYk3cuKmYTmJGgE0nkFEJbpVs3q1G5+P9zu52Z7tOy2WDCbC6ah7MpcvBR0olAz/kkzROwHVcBzWcm47/KY+ZizsgQpqCK00/SRVRzZgS9xys8xnSBYhIOuYMJkDEYjLgiSCuqcy5lE4FnJvxOSQewccI80NjsYxVbwNhMCcwGUAUxWxEMj4XREw4l+Nk0eh2MMu9D67tdq52dJs5Y+SaQ9XHrvpYUR9hoj5/v/2D+mqqkBoRDMkIZCAwsiCOhOuRlS2SxEL2Yib7LV0y0/BwKPN+t6Fje6ttfAskR21VM9Eq6yPq1cm1fPuAFVLq6BUVe4WVnuhrA1q7nlg8NKuFbF9byEBOM6YNoV5vtW8T0wdfFBSgBXOIR2Ppmq+WOcP8fWCCZxithaBVpk9uEjxlAcNTXxqaqo5BB0/KjnYkSA9PsibZ7lfqNKEyiw9VKXHzXbXZJGkGArJZUcZVdR6vEX2Yy3iiz16NzJCYUH0xRophYmSrTU5htQmkAoVRy43kmZFY6ERK8WKnfYIFWmKGxFNF7dmKaugLW0Nc2DwZYD+vgNIfUAHBDEI0YcR62mLfHyacStfzJ0CZG8WT9mqTHACk6nE/m4JxP8Ix1raAXwamSMQzN4JZHELbCdOpgxYom9JEB+6ur66fN5qMxwIK3Qxrw1x3qcub9eDX+rgyyv208alJIrlIoW3M6KA31j3yFlmDlQ3jCOaBLiG2XJ6tj5G5qut1FJ4veR6zdWfecuP5onrxqnQ2XkMEqmdkmoU75JBGzasTfZTKYMIjSFz92cK57F/l0TQBb4mq+9jO611A/GQcbkOICplPs0C1edkxpvWR1YrAyF9kwEqa0BAKjqIoJq/9Lav7c55V+k/khMQVH4+Fys4bErsAfY5OX9XeynqOKpXUK5RMrAhKJ1dxT/HZ1OGVVhFphNtUsGq6VsXqRllJ67QaDQ6SdBG4Ogcj9IbYqciNuRzFMRVUyswUG090O1gCffULBM4MxysBNG6WZcipwNeJJfgslsJyi+FBMXott7oLJsegjGf7GfOBhUHq/zdiaWenWvifS6dw6dVwVehjShsznEEoMAOGpcVpxw0BFQxYCeaqtxYREkejerdjMYpDicePUauumek4TRPoaZ1yp18QZEf7FGRT0tGW+aEiCJXGGrqieLfgw6EAlME4iBjHQ3wUWBsy4FMW0SwG3ApD7A1MKlmUs9Kaxmllho+aWDDTNxAdqx5M/ZIE6vLYJrW7t48z0IWZ/nIOYhY5nlfII9UYVz+BDkob6i/keIKyKRSLKVdRTOihi1NtgndENXgUmDheEObCgU4Xb2Krnld1Y/LwaZoCi1wXremLrhWxUVippbmv3JhyNaqo+HhZke4BLNoJnQwiSg5b5BCntLHJpzJIaQ08C3xbjXKcd2UKONbWjFY4zdAqiqzqV7zTzAKTulmgUQRR8aaqkaeiarKUwHyMt+vc6KaSrcNsIuoZgT7a1fbrlTDKZ9tkrVgvcssBVVn08mBb+N/3XiOMuJd1LTPEz2LZ5IlONUZVBRuy8lGL919isgHVAbHmVQlajTfEpIpHcYToa3wbm9H8filiKQ8ZxZ56G4lUlUwfAqVvFBFpz7K4j7yw0VQW+4U0WGnM9qSsXiolz5AreBE0Lb9qjgR1euEhsarITwkeHWROhWUQQk9VO0RwaJlW6ZI8mxNMKkiZZ4z5tYk6qlyReid2ygwxg6U9kw+Oj9qO8pnX1kiWzbsUUr7SW1nrlwZVtfNy1W6F+WKzUGz8A+m0PUUMEgAA', 'phrase-overrides.json': 'H4sIAAAAAAACCo1UTU/bQBC9R+p/iHzmYHu9+ejPKEfUQ6S6JRUNyEmQEELKbFBJAhUFKrVSRWkFCQWEOCGkkMJ/YRLwrX+hu9lNvGuHpJIPPrw3897Mm11PpdPW/CvbcWyatV6mF6zHb71+bw/h3ppLWyW/WgkKS+K3+GGlUCn6pYr1ei4iZe0hCdkmwk8BK1eDlaBY9sV/ofTODwx4zh3Cw+sbhE/ItuOMUT+NQxzJwfou1i8E6P3ymsHTwa7U83QKCDsCFfjl5aXVyXKIR2Vp+IIMkB081Dr8i4tarAaGaUIzinaJsDUL7XleJOkq0qFp09FUGgh/1xCOEY7GJScZ8Kgja18zZLvIWjr6rV+IgYnaFUP4+NgcDr/il95IMQndGak7hB8I2wjfETp6+Yl58PJSUP+2Neh2lN/y4tpobQZU3yuy/Rly8lJ9WPuM0EDWRLaDcI6wOXVC1KF6lxqYXeIzoq6TFKX0xxVRN6tjpdfnhk8zxJimEI5wqpVXosYc1x5dC8IZXxjCV7WBKScmSLmZJNMJESzNyd/ekZFS0wpHu3nV4XDYodXvdgftP7xVvE/cEucSlUHh/gThjs+A53F6P8/5/6AIuB7z8J4/ZW2EPZOUuDzOy5FocAcIF4ao5GMjGLlkAJ61QYiMy6DdeLrZR/iFrM6fnDE0uUxqOyPrPC+sOWhsqSczuj3dS2rjReofzFtChNAFAAA='}
for name, payload in embedded.items():
    (job / name).write_bytes(gzip.decompress(base64.b64decode(payload)))


In [ ]:
EPOCHS = 5
BATCH_SIZE = 16
RELEASE_GPU = True

env = {**os.environ, 'SORAVOICE_APP': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
cmd = [
    sys.executable, '-u', str(job / 'train_sovits.py'),
    '--data', str(job / 'training-data-v3.pt'),
    '--prep', str(job / 'prep'),
    '--base-g', str(job / full_checkpoint),
    '--init-g-export', str(job / epoch25),
    '--out', str(out),
    '--cross-ref-manifest', str(job / 'cross-reference-manifest.json'),
    '--freeze-backbone', '--unfreeze-flow',
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
]
def run_training(command):
    with subprocess.Popen(
        command, env=env, cwd=repo, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

run_training(cmd + ['--smoke'])
run_training(cmd)

from io import BytesIO
import torch
sys.path[:0] = [str(repo / 'GPT_SoVITS'), str(repo)]
weight_path = out / f'sora-sovits-v2proplus-epoch{EPOCHS}.pth'
with weight_path.open('rb') as stream:
    header = stream.read(2)
    if header != b'06':
        raise ValueError(f'SoVITS 权重文件头错误：{weight_path}')
    state = torch.load(BytesIO(b'PK' + stream.read()), map_location='cpu', weights_only=False)
assert state.get('weight') and state.get('emotion_adapter'), 'SoVITS 权重不全'
assert state['info'].startswith(f'{EPOCHS}epoch_')
resume = out / 's2_v2proplus_emotion_resume.pt'
assert resume.is_file() and resume.stat().st_size > weight_path.stat().st_size, '续训断点缺失'
print(f'权重和断点已保存到 {out}', flush=True)
if RELEASE_GPU:
    drive.flush_and_unmount()
    runtime.unassign()